**Configuración, Conexión y Contexto de Spark**

In [ ]:
from pyspark import SparkContext

sc = SparkContext("local", "MiAplicacionSpark")

print(sc)

<SparkContext master=local appName=MiAplicacionSpark>


**Creación y Carga de un RDD**

1. Desde una colección en Python

In [ ]:
datos = [1, 2, 3, 4, 5]
rdd = sc.parallelize(datos)

2. Desde una base de datos o fuente externa

In [ ]:
rdd = sc.textfile("ruta/file.txt")

In [ ]:
rdd_filtrado = rdd.filter(lambda x: x > 10)

rdd_filtrado.count()
rdd_filtrado.collect()

**Almacenamiento de un RDD**

* Sin cache

In [ ]:
from pyspark import SparkContext
# sc = SparkContext("local", "Ejemplo")

datos = [5, 15, 25, 8, 30]

rdd = sc.parallelize(datos)

rdd_filtrado = rdd.filter(lambda x: x > 10)

print(rdd_filtrado.count())    # acción 1
print(rdd_filtrado.collect())  # acción 2

3
[15, 25, 30]


* Con cache

In [ ]:
rdd_filtrado = rdd.filter(lambda x: x > 10)

rdd_filtrado.cache()

print(rdd_filtrado.count())    # ejecuta y guarda
print(rdd_filtrado.collect())  # usa lo guardado

3
[15, 25, 30]


Qué cambia realmente

Primera acción → Spark calcula y guarda

Segunda acción → Spark reutiliza

In [ ]:
rdd = sc.parallelize([("a", 1), ("b", 2), ("a", 3)])

In [ ]:
rdd.reduceByKey(lambda x, y: x + y).collect()

[('a', 4), ('b', 2)]

**Ejemplo de Transformaciones**

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4, 5])

resultado = rdd.filter(lambda x: x % 2 == 0) \
               .map(lambda x: x**2)

print(resultado.collect())  # [4, 16]

[4, 16]


**Ejemplo de Acciones**

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4, 5])

print(rdd.sum())     # 15
print(rdd.take(3))   # [1, 2, 3]

15
[1, 2, 3]


**Ejemplo de reduceByKey**

In [ ]:
# Crear un Pair-RDD
pairs = sc.parallelize([("a", 1), ("b", 1), ("a", 1), ("a", 1), ("b", 1)])

# Sumar valores por clave
sums = pairs.reduceByKey(lambda a, b: a + b)

# Resultado esperado: [("a", 3), ("b", 2)]
print(sums.collect())

[('a', 3), ('b', 2)]


# **PARTE 2**

In [ ]:
# Crear variable broadcastbroadcast
Var = sc.broadcast([1, 2, 3])
# Acceder al valorbroadcast
Var.value


[1, 2, 3]

**Acumuladores**

In [ ]:
# Crear acumulador
accum = sc.accumulator(0)

# Función que incrementa el acumulador
def f(x):
    global accum
    accum += x
    return x

# Aplicar función a cada elemento
rdd.foreach(f)

# Valor final del acumulador
print(accum.value)

**Particionamiento Personalizado**

In [ ]:
# Crear un RDD con particionamiento personalizado
rdd = pairs.partitionBy(4, lambda key: hash(key) % 4)

**Spark SQL: Integración con RDDs**

In [ ]:
# Crear DataFrame desde RDD
from pyspark.sql import Row

Person = Row("name", "age")
people = rdd.map(lambda r: Person(r[0], r[1]))

df = spark.createDataFrame(people)

# Registrar como tabla temporal
df.createOrReplaceTempView("people")

# Ejecutar consulta SQL
result = spark.sql("SELECT * FROM people WHERE age > 20")

**Spark Streaming: Procesamiento en Tiempo Real**

In [ ]:
from pyspark import SparkContext
from pyspark.streaming import StreamingContext

# Crear SparkContext
sc = SparkContext("local[2]", "StreamingDemo")

# Crear contexto de streaming con intervalo de 1 segundo
ssc = StreamingContext(sc, 1)

# Crear DStream desde socket
lines = ssc.socketTextStream("localhost", 9999)

# Contar palabras
words = lines.flatMap(lambda line: line.split(" "))
pairs = words.map(lambda word: (word, 1))
wordCounts = pairs.reduceByKey(lambda a, b: a + b)

# Imprimir resultados
wordCounts.pprint()

# Iniciar procesamiento
ssc.start()
ssc.awaitTermination()

**MLlib: Biblioteca de Machine Learning**

In [ ]:
from pyspark.mllib.classification import LogisticRegressionWithSGD
from pyspark.mllib.regression import LabeledPoint

# Crear datos de entrenamiento
data = [
    LabeledPoint(0.0, [0.0, 1.0]),
    LabeledPoint(1.0, [1.0, 0.0])
]

rdd = sc.parallelize(data)

# Entrenar modelo
model = LogisticRegressionWithSGD.train(rdd)

# Predecir
print(model.predict([1.0, 0.0]))

**Caso Práctico: Análisis de Logs**

In [ ]:
# Cargar logs desde archivo
logs = sc.textFile("logs.txt")

# Filtrar líneas que contienen errores
errors = logs.filter(lambda line: "ERROR" in line)

# Extraer códigos de error
error_codes = errors.map(lambda line: line.split(" ")[3])

# Contar ocurrencias por código
error_counts = error_codes.map(lambda code: (code, 1)) \
                          .reduceByKey(lambda a, b: a + b)

# Ordenar por frecuencia descendente
sorted_errors = error_counts.sortBy(lambda x: -x[1])

# Mostrar los 10 errores más comunes
print(sorted_errors.take(10))

**Caso Práctico: Análisis de Texto**

In [ ]:
# Cargar texto desde archivo
text = sc.textFile("libro.txt")

# Dividir cada línea en palabras y convertirlas a minúsculas
words = text.flatMap(lambda line: line.lower().split(" "))

# Filtrar palabras vacías y dejar solo palabras alfabéticas
words = words.filter(lambda word: len(word) > 0 and word.isalpha())

# Contar frecuencia de cada palabra
word_counts = words.map(lambda word: (word, 1)) \
                   .reduceByKey(lambda a, b: a + b)

# Ordenar por frecuencia descendente
sorted_counts = word_counts.sortBy(lambda x: -x[1])

# Mostrar las 20 palabras más comunes
print(sorted_counts.take(20))

**Caso Práctico: Análisis de Datos CSV**

In [ ]:
# Importar SparkSession
from pyspark.sql import SparkSession

# Crear o recuperar la sesión de Spark
spark = SparkSession.builder.getOrCreate()

# Leer archivo CSV
df = spark.read.csv(
    "ventas.csv",
    header=True,
    inferSchema=True
)

# Registrar el DataFrame como tabla temporal
df.createOrReplaceTempView("ventas")

# Ejecutar consulta SQL para analizar ventas por región
result = spark.sql("""
    SELECT
        region,
        SUM(monto) AS total_ventas
    FROM ventas
    GROUP BY region
    ORDER BY total_ventas DESC
""")

# Mostrar resultados
result.show()